# N8n MCP Agent Validation Notebook

This notebook is for testing and validating the `N8nMCPAgent`. It allows for rapid iteration on the workflow generation logic.

## 1. Setup

Import the agent and other necessary libraries.

In [ ]:
import sys
import os
import json

# Add the parent directory to the system path to allow importing the agent
sys.path.insert(0, os.path.abspath('..'))

from agents.n8n_mcp_agent import N8nMCPAgent

# Suppress informational logging for cleaner output
import logging
logging.basicConfig(level=logging.WARNING)

print("Setup complete. N8nMCPAgent imported.")

## 2. Instantiate the Agent

Create an instance of the agent. We can configure the n8n URL and API key here if needed.

In [ ]:
agent = N8nMCPAgent()
print(f"Agent initialized for n8n instance at: {agent.n8n_base_url}")

## 3. Generate the "Dynamic Content Pipeline" Workflow

This cell simulates the full process:
1. A TOON plan is provided (currently hardcoded inside the agent).
2. The agent parses the plan.
3. The agent generates the n8n workflow from the plan.

In [ ]:
# 1. Define the TOON plan (using a dummy string as the agent's parser is hardcoded)
toon_plan_full = """
name: Dynamic Content Pipeline
trigger: schedule(every=6h)
steps: [
  fetch.rss(url='https://arxiv.org/rss/cs.AI'),
  call.agent.summarizer(content=$fetch.items[0]),
  call.agent.creative(concept=$summarizer.key_concept),
  ai.image.comfyui(prompt=$creative.image_prompt),
  call.agent.writer(topic=$summarizer.summary),
  post.wordpress(title=$writer.title, content=$writer.post, image=$comfyui.image),
  post.twitter(text=$writer.tweet, media=$comfyui.image)
]
"""

# 2. Parse the plan
structured_plan = agent.parse_toon_plan(toon_plan_full)
print("--- Structured Plan (from TOON parser) ---")
print(json.dumps(structured_plan, indent=2))

# 3. Generate the workflow
workflow_json = agent.generate_n8n_workflow(structured_plan)
print("\n--- Generated n8n Workflow JSON ---")
print(workflow_json)

## 4. Validate and Inspect the Workflow

We can now validate the generated JSON and inspect its structure.

In [ ]:
is_valid = agent.validate_workflow(workflow_json)
print(f"Workflow validation successful: {is_valid}")

if is_valid:
    workflow_data = json.loads(workflow_json)
    print(f"\nWorkflow Name: {workflow_data['name']}")
    print(f"Number of nodes: {len(workflow_data['nodes'])}")
    print(f"Number of connections: {len(workflow_data['connections'])}")
    
    # You can now copy the JSON output from the cell above and paste it 
    # into the n8n editor using 'Import from Clipboard' to manually verify.